# 23 — LCEL (LangChain Expression Language)

Compose chains with pipes, parallel execution, branching, and lambdas.

In [ ]:
import os
os.environ['OPENAI_API_KEY'] = 'your-key'

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda, RunnableBranch

## Example 1: Basic Pipe Composition

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = ChatPromptTemplate.from_template("Explain {concept} in one sentence for a {audience}.") | llm | StrOutputParser()

for inp in [{"concept": "recursion", "audience": "5-year-old"}, {"concept": "recursion", "audience": "senior engineer"}]:
    print(f"[{inp['audience']}] {chain.invoke(inp)}")

## Example 2: RunnableParallel

In [ ]:
pros_chain = ChatPromptTemplate.from_template("List 2 pros of {technology}. Be brief.") | llm | StrOutputParser()
cons_chain = ChatPromptTemplate.from_template("List 2 cons of {technology}. Be brief.") | llm | StrOutputParser()
parallel = RunnableParallel(pros=pros_chain, cons=cons_chain)

for tech in ["microservices", "monoliths"]:
    result = parallel.invoke({"technology": tech})
    print(f"{tech}:\n  Pros: {result['pros'][:80]}...\n  Cons: {result['cons'][:80]}...\n")

## Example 3: RunnableBranch

In [ ]:
math_chain = ChatPromptTemplate.from_template("Solve step by step:\n{input}") | llm | StrOutputParser()
code_chain = ChatPromptTemplate.from_template("Write Python code:\n{input}") | llm | StrOutputParser()
general_chain = ChatPromptTemplate.from_template("Answer concisely:\n{input}") | llm | StrOutputParser()

def classify(inp):
    text = inp["input"].lower()
    if any(w in text for w in ["calculate", "solve", "math", "sum"]): return "math"
    if any(w in text for w in ["code", "function", "program", "implement"]): return "code"
    return "general"

branch = RunnableBranch(
    (lambda x: classify(x) == "math", math_chain),
    (lambda x: classify(x) == "code", code_chain),
    general_chain,
)

for q in ["Calculate the sum of first 10 prime numbers", "Write a function to check palindrome", "Capital of Australia?"]:
    cat = classify({"input": q})
    print(f"[{cat}] Q: {q}\nA: {branch.invoke({'input': q})[:120]}...\n")

## Example 4: RunnableLambda

In [ ]:
def preprocess(d):
    text = d["text"]
    return {"text": text.strip().lower(), "word_count": len(text.split()), "char_count": len(text)}

def postprocess(result):
    return {"summary": result, "summary_length": len(result)}

chain = (
    RunnableLambda(preprocess)
    | ChatPromptTemplate.from_template("Summarise this text ({word_count} words) in one sentence:\n\n{text}")
    | llm | StrOutputParser()
    | RunnableLambda(postprocess)
)

result = chain.invoke({"text": "  LangChain Expression Language allows you to compose chains using the pipe operator. It supports parallel execution, branching, and custom transformations.  "})
print(f"Summary: {result['summary']}\nLength: {result['summary_length']} chars")